# 02 数据清洗

参考 `utils/data_cleaner.py` 的清洗逻辑，演示完整清洗流程：
1. 只保留 delivered 状态的订单
2. 解析时间字段
3. 删除关键字段缺失的行
4. 关联 customers 表获取唯一用户标识
5. 计算物流衍生字段

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

from utils.db_connector import get_engine
engine = get_engine()

## 2.1 加载原始数据

In [ ]:
# 读取原始订单数据
with engine.connect() as conn:
    orders_raw = pd.read_sql_query("SELECT * FROM olist_orders", conn)

print(f"原始订单数: {len(orders_raw):,}")
print(f"\n订单状态分布:")
print(orders_raw['order_status'].value_counts())
orders_raw.head()

## 2.2 过滤 - 只保留已送达订单

In [ ]:
# 只保留 delivered 订单
orders_delivered = orders_raw[orders_raw['order_status'] == 'delivered'].copy()
print(f"过滤后订单数: {len(orders_delivered):,}")
print(f"过滤掉: {len(orders_raw) - len(orders_delivered):,} 条非 delivered 订单")

## 2.3 解析时间字段

In [ ]:
# 时间字段解析
datetime_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in datetime_cols:
    if col in orders_delivered.columns:
        orders_delivered[col] = pd.to_datetime(orders_delivered[col], errors='coerce')

print("时间字段解析完成")
print(f"\n时间范围: {orders_delivered['order_purchase_timestamp'].min()} ~ {orders_delivered['order_purchase_timestamp'].max()}")
orders_delivered[datetime_cols].dtypes

## 2.4 删除关键字段缺失的行

In [ ]:
# 关键字段缺失值检查
critical_cols = ['customer_id', 'order_id', 'order_purchase_timestamp']

before_drop = len(orders_delivered)
orders_clean = orders_delivered.dropna(subset=critical_cols).reset_index(drop=True)
after_drop = len(orders_clean)

print(f"删除关键字段为空的行: {before_drop - after_drop} 条")
print(f"剩余订单数: {after_drop:,}")

## 2.5 关联 customers 表获取 customer_unique_id

In [ ]:
# 关联 customers 表
with engine.connect() as conn:
    customers = pd.read_sql_query(
        "SELECT customer_id, customer_unique_id FROM olist_customers", conn
    )

orders_clean = orders_clean.merge(customers, on='customer_id', how='left')

missing_uid = orders_clean['customer_unique_id'].isna().sum()
print(f"未匹配到 customer_unique_id 的行数: {missing_uid}")
print(f"唯一用户数: {orders_clean['customer_unique_id'].nunique():,}")

## 2.6 计算物流衍生字段

In [ ]:
# 物流延迟天数 = 实际送达 - 预计送达
orders_clean['delivery_delay_days'] = (
    orders_clean['order_delivered_customer_date'] - orders_clean['order_estimated_delivery_date']
).dt.days

# 实际配送天数 = 实际送达 - 购买时间
orders_clean['delivery_days'] = (
    orders_clean['order_delivered_customer_date'] - orders_clean['order_purchase_timestamp']
).dt.days

print(f"平均配送天数: {orders_clean['delivery_days'].mean():.1f} 天")
print(f"平均延迟天数: {orders_clean['delivery_delay_days'].mean():.1f} 天")
print(f"延迟订单占比: {(orders_clean['delivery_delay_days'] > 0).mean():.1%}")

## 2.7 清洗前后对比

In [ ]:
# 清洗前后对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 清洗漏斗
stages = ['原始数据', '仅delivered', '去除缺失', '最终数据']
counts = [len(orders_raw), len(orders_delivered), after_drop, len(orders_clean)]
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']

axes[0].barh(stages, counts, color=colors, alpha=0.8)
for i, v in enumerate(counts):
    axes[0].text(v + 100, i, f'{v:,}', va='center')
axes[0].set_xlabel('订单数')
axes[0].set_title('数据清洗漏斗')

# 配送天数分布
valid_delivery = orders_clean['delivery_days'].dropna()
axes[1].hist(valid_delivery, bins=50, color='#3498db', alpha=0.7, edgecolor='white')
axes[1].axvline(valid_delivery.median(), color='red', linestyle='--', label=f'中位数: {valid_delivery.median():.0f}天')
axes[1].set_xlabel('配送天数')
axes[1].set_ylabel('订单数')
axes[1].set_title('配送天数分布')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n=== 清洗统计摘要 ===")
print(f"原始订单数:   {len(orders_raw):,}")
print(f"清洗后订单数: {len(orders_clean):,}")
print(f"保留比例:     {len(orders_clean)/len(orders_raw):.1%}")
print(f"唯一用户数:   {orders_clean['customer_unique_id'].nunique():,}")

## 小结

- 数据清洗保留了约 96% 的已送达订单
- 配送平均耗时约12天，约 8% 的订单存在延迟
- `customer_unique_id` 为用户唯一标识，后续分析基于此字段聚合